# 🩺 Breast Ultrasound Classification
## EfficientNet-B0 + Transfer Learning | Google Colab

**Anti-Overfitting:** Transfer Learning, Dropout, Label Smoothing, Weight Decay, Early Stopping, Online Augmentation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Imports & Setup

In [ ]:
import os
import copy
import time
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 2. Configuration

In [ ]:
class Config:
    DATASET_ROOT = "/content/drive/MyDrive/Breast_Dataset/Dataset_BUSI_with_GT_Split"
    OUTPUT_DIR = "/content/breast_ultrasound_results"

    MODEL_NAME = "efficientnet_b0"
    NUM_CLASSES = 3
    CLASS_NAMES = ["benign", "malignant", "normal"]

    IMG_SIZE = 256
    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD = [0.229, 0.224, 0.225]

    STAGE1_EPOCHS = 15
    STAGE1_LR = 1e-3
    STAGE1_BATCH_SIZE = 32

    STAGE2_EPOCHS = 30
    STAGE2_LR = 1e-4
    STAGE2_BATCH_SIZE = 32
    STAGE2_UNFREEZE_FROM = 6

    WEIGHT_DECAY = 1e-4
    DROPOUT = 0.4
    LABEL_SMOOTHING = 0.1
    PATIENCE = 8
    SEED = 42
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(Config.SEED)
os.makedirs(Config.OUTPUT_DIR, exist_ok=True)
print(f"Device: {Config.DEVICE}")


## 3. Dataset & Transforms

In [ ]:
class BreastUltrasoundDataset(Dataset):
    def __init__(self, root_dir, split, transform=None):
        self.root_dir = root_dir
        self.split = split
        self.transform = transform
        self.class_names = Config.CLASS_NAMES
        self.class_to_idx = {name: idx for idx, name in enumerate(self.class_names)}
        self.samples = []

        split_dir = os.path.join(root_dir, split)
        for class_name in self.class_names:
            class_dir = os.path.join(split_dir, class_name)
            if not os.path.exists(class_dir):
                continue
            for sample_id in os.listdir(class_dir):
                img_path = os.path.join(class_dir, sample_id, "image.png")
                if os.path.exists(img_path):
                    self.samples.append((img_path, self.class_to_idx[class_name]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

    def get_class_distribution(self):
        labels = [label for _, label in self.samples]
        return Counter(labels)


In [ ]:
def get_transforms():
    train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15, interpolation=transforms.InterpolationMode.BILINEAR, fill=128),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
        transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
        transforms.ToTensor(),
        transforms.Normalize(Config.IMAGENET_MEAN, Config.IMAGENET_STD),
        transforms.RandomErasing(p=0.1, scale=(0.02, 0.1)),
    ])

    val_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(Config.IMAGENET_MEAN, Config.IMAGENET_STD),
    ])

    return train_transform, val_transform


## 4. Model

In [ ]:
def build_model():
    weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
    model = models.efficientnet_b0(weights=weights)

    for param in model.parameters():
        param.requires_grad = False

    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=Config.DROPOUT),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.BatchNorm1d(256),
        nn.Dropout(p=Config.DROPOUT * 0.5),
        nn.Linear(256, Config.NUM_CLASSES),
    )
    return model.to(Config.DEVICE)


def unfreeze_backbone(model, from_block=6):
    for i, block in enumerate(model.features):
        if i >= from_block:
            for param in block.parameters():
                param.requires_grad = True
    return model


## 5. Training Loop

In [ ]:
class EarlyStopping:
    def __init__(self, patience=8, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.should_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop


def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


def train_stage(model, train_loader, val_loader, criterion, optimizer, scheduler,
                num_epochs, device, stage_name, early_stopping):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_acc = 0.0

    print(f"\n{'=' * 50}")
    print(f"  {stage_name}")
    print(f"{'=' * 50}")
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_p = sum(p.numel() for p in model.parameters())
    print(f"  Trainable: {trainable:,} / {total_p:,} ({100*trainable/total_p:.1f}%)")
    print(f"  Param/Sample: {trainable/len(train_loader.dataset):.1f}x\n")

    for epoch in range(num_epochs):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        if scheduler:
            scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        elapsed = time.time() - t0
        marker = ""
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            marker = " ★ best"

        print(f"  Epoch {epoch+1:02d}/{num_epochs} | "
              f"Train: {train_loss:.4f} / {train_acc:.4f} | "
              f"Val: {val_loss:.4f} / {val_acc:.4f} | "
              f"{elapsed:.1f}s{marker}")

        if early_stopping(val_loss):
            print(f"\n  ⚠ Early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(best_model_wts)
    print(f"\n  Best Val Accuracy: {best_val_acc:.4f}")
    return model, history


## 6. Evaluation & Visualization

In [ ]:
@torch.no_grad()
def get_predictions(model, dataloader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for images, labels in dataloader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())
    return np.array(all_preds), np.array(all_labels), np.array(all_probs)


def plot_training_history(history_stage1, history_stage2, save_path):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("Training History", fontsize=16, fontweight="bold")
    stages = [(history_stage1, "Stage 1: Head Only"), (history_stage2, "Stage 2: Fine-Tune")]
    for col, (history, title) in enumerate(stages):
        epochs = range(1, len(history["train_loss"]) + 1)
        axes[0, col].plot(epochs, history["train_loss"], "b-o", label="Train", markersize=3)
        axes[0, col].plot(epochs, history["val_loss"], "r-o", label="Val", markersize=3)
        axes[0, col].set_title(f"{title} — Loss"); axes[0, col].set_xlabel("Epoch")
        axes[0, col].set_ylabel("Loss"); axes[0, col].legend(); axes[0, col].grid(True, alpha=0.3)
        axes[1, col].plot(epochs, history["train_acc"], "b-o", label="Train", markersize=3)
        axes[1, col].plot(epochs, history["val_acc"], "r-o", label="Val", markersize=3)
        axes[1, col].set_title(f"{title} — Accuracy"); axes[1, col].set_xlabel("Epoch")
        axes[1, col].set_ylabel("Accuracy"); axes[1, col].legend(); axes[1, col].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, "training_history.png"), dpi=150, bbox_inches="tight")
    plt.show()


def plot_confusion_matrix(y_true, y_pred, class_names, save_path):
    cm = confusion_matrix(y_true, y_pred)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=axes[0], cmap="Blues", values_format="d")
    axes[0].set_title("Confusion Matrix (Counts)")
    cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
    ConfusionMatrixDisplay(cm_norm, display_labels=class_names).plot(ax=axes[1], cmap="Blues", values_format=".2f")
    axes[1].set_title("Confusion Matrix (Normalized)")
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, "confusion_matrix.png"), dpi=150, bbox_inches="tight")
    plt.show()


def plot_per_class_metrics(y_true, y_pred, class_names, save_path):
    report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
    metrics = ["precision", "recall", "f1-score"]
    data = {m: [report[c][m] for c in class_names] for m in metrics}
    x = np.arange(len(class_names)); width = 0.25
    fig, ax = plt.subplots(figsize=(8, 5))
    for i, metric in enumerate(metrics):
        bars = ax.bar(x + i * width, data[metric], width, label=metric.capitalize())
        for bar, val in zip(bars, data[metric]):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.01, f"{val:.2f}", ha="center", fontsize=9)
    ax.set_ylabel("Score"); ax.set_title("Per-Class Metrics")
    ax.set_xticks(x + width); ax.set_xticklabels(class_names)
    ax.legend(); ax.set_ylim(0, 1.15); ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, "per_class_metrics.png"), dpi=150, bbox_inches="tight")
    plt.show()


## 7. 🚀 Eğitimi Başlat

In [ ]:
def main():
    print("=" * 60)
    print("  BREAST ULTRASOUND CLASSIFICATION")
    print("=" * 60)
    print(f"  Device: {Config.DEVICE}")

    # Data
    print("\n[1/5] Loading dataset...")
    train_tf, val_tf = get_transforms()
    train_ds = BreastUltrasoundDataset(Config.DATASET_ROOT, "train", train_tf)
    val_ds = BreastUltrasoundDataset(Config.DATASET_ROOT, "val", val_tf)
    test_ds = BreastUltrasoundDataset(Config.DATASET_ROOT, "test", val_tf)
    print(f"  Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

    train_loader = DataLoader(train_ds, batch_size=Config.STAGE1_BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=Config.STAGE1_BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=Config.STAGE1_BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    # Model
    print("\n[2/5] Building model...")
    model = build_model()
    criterion = nn.CrossEntropyLoss(label_smoothing=Config.LABEL_SMOOTHING)

    # Stage 1
    print("\n[3/5] Stage 1: Training classifier head...")
    opt1 = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=Config.STAGE1_LR, weight_decay=Config.WEIGHT_DECAY)
    sch1 = optim.lr_scheduler.ReduceLROnPlateau(opt1, mode="min", factor=0.5, patience=3)
    model, hist1 = train_stage(model, train_loader, val_loader, criterion, opt1, sch1, Config.STAGE1_EPOCHS, Config.DEVICE, "STAGE 1 — Head Only", EarlyStopping(Config.PATIENCE))

    # Stage 2
    print("\n[4/5] Stage 2: Fine-tuning backbone...")
    model = unfreeze_backbone(model, from_block=Config.STAGE2_UNFREEZE_FROM)
    opt2 = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=Config.STAGE2_LR, weight_decay=Config.WEIGHT_DECAY)
    sch2 = optim.lr_scheduler.ReduceLROnPlateau(opt2, mode="min", factor=0.5, patience=3)
    model, hist2 = train_stage(model, train_loader, val_loader, criterion, opt2, sch2, Config.STAGE2_EPOCHS, Config.DEVICE, "STAGE 2 — Fine-Tune Backbone", EarlyStopping(Config.PATIENCE))

    # Test
    print("\n[5/5] Evaluating on TEST set...")
    test_loss, test_acc = evaluate(model, test_loader, criterion, Config.DEVICE)
    print(f"\n  TEST ACCURACY: {test_acc:.4f} | TEST LOSS: {test_loss:.4f}")

    y_pred, y_true, y_probs = get_predictions(model, test_loader, Config.DEVICE)
    print("\n" + classification_report(y_true, y_pred, target_names=Config.CLASS_NAMES))

    # Plots
    plot_training_history(hist1, hist2, Config.OUTPUT_DIR)
    plot_confusion_matrix(y_true, y_pred, Config.CLASS_NAMES, Config.OUTPUT_DIR)
    plot_per_class_metrics(y_true, y_pred, Config.CLASS_NAMES, Config.OUTPUT_DIR)

    # Save
    torch.save({"model_state_dict": model.state_dict(), "class_names": Config.CLASS_NAMES, "test_accuracy": test_acc}, os.path.join(Config.OUTPUT_DIR, "best_model.pth"))
    print(f"\n  Model saved to {Config.OUTPUT_DIR}/best_model.pth")

    # Copy to Drive
    import shutil
    drive_out = os.path.join(os.path.dirname(Config.DATASET_ROOT), "classification_results")
    if os.path.exists(os.path.dirname(Config.DATASET_ROOT)):
        if os.path.exists(drive_out):
            shutil.rmtree(drive_out)
        shutil.copytree(Config.OUTPUT_DIR, drive_out)
        print(f"  Results copied to Drive: {drive_out}")

main()
